<a href="https://colab.research.google.com/github/mcundins/cse6242-project/blob/KMeansModel/kmeans_modeling/kmeans_food_desert_risk_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# K-means clustering (unsupervised learning)

Testing model with 2025 dataset

In [18]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, confusion_matrix
import pickle

In [19]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [20]:
with open("/content/drive/MyDrive/Food_Desert_Data/cluster_model.pkl", "rb") as f:
    bundle = pickle.load(f)
pipe, model_cols, id_col = bundle["pipeline"], bundle["model_cols"], bundle["id_col"]

df_2025 = pd.read_csv("/content/drive/MyDrive/Food_Desert_Data/cleaned_2025.csv", dtype={id_col: str})


In [21]:
df_2025.head()

,tract_id,State,County,meta_dataset_version,meta_population_census_year,meta_tract_vintage,fd_label,label_status,label_reason,original__Urban,...,income_with_endpoint_proxy_2018_usd,income_endpoint_proxy_applied,income_proxy_method,income_ordinary_reference_candidate,FD_2_original,FD_2,FD_2_source_field,FD_2_status,FD_2_reason,FD_2_comparison
0,01001020100,Alabama,Autauga County,2025,2020,2020,1.0,known,both_thresholds,1,...,61431.919806,0,ordinary_amount_unchanged,1,1.0,1.0,SD_SRAM_LA1and20,known,official_flag_available,agree
1,01001020200,Alabama,Autauga County,2025,2020,2020,0.0,known,below_both_thresholds,1,...,49742.306258,0,ordinary_amount_unchanged,1,0.0,0.0,SD_SRAM_LA1and20,known,official_flag_available,agree
2,01001020300,Alabama,Autauga County,2025,2020,2020,1.0,known,count_threshold_only,1,...,67187.346684,0,ordinary_amount_unchanged,1,1.0,1.0,SD_SRAM_LA1and20,known,official_flag_available,agree
3,01001020400,Alabama,Autauga County,2025,2020,2020,0.0,known,below_both_thresholds,1,...,64749.350412,0,ordinary_amount_unchanged,1,0.0,0.0,SD_SRAM_LA1and20,known,official_flag_available,agree
4,01001020501,Alabama,Autauga County,2025,2020,2020,0.0,known,below_both_thresholds,1,...,86373.699013,0,ordinary_amount_unchanged,1,0.0,0.0,SD_SRAM_LA1and20,known,official_flag_available,agree


In [22]:
df_2025.isna().sum()

,0
tract_id,0
State,0
County,0
meta_dataset_version,0
meta_population_census_year,0
...,...
FD_2,271
FD_2_source_field,0
FD_2_status,0
FD_2_reason,0


## Preprocessing

In [23]:
df_2025 = df_2025.iloc[:, :-4]
df_2025 = df_2025[[i for i in df_2025.columns if 'original' not in i]]
df_2025 = df_2025[[i for i in df_2025.columns if 'qa' not in i]]
df_2025 = df_2025[[i for i in df_2025.columns if 'income' not in i]]
df_2025 = df_2025.drop(columns=['fd_label', 'label_status', 'label_reason'])

#leading 0
df_2025['tract_id'] = df_2025['tract_id'].astype(str).str.zfill(11)

#renaming 'State' as state_name
df_2025 = df_2025.rename(columns={'State': 'state_name'})

#creating state column with 2 beg digits of tract_id
df_2025['state'] = df_2025['tract_id'].str[:2]

df_2025.head()

,tract_id,state_name,County,meta_dataset_version,meta_population_census_year,meta_tract_vintage,clean__Urban,clean__population_total,clean__occupied_housing_units_total,clean__GroupQuartersFlag,...,clean__MedianFamilyIncome,clean__TractLOWI,clean__TractHUNV,clean__TractSNAP,clean__LATracts_half,clean__LATracts1,clean__LATractsVehicle_20,clean__low_access_population_1_20,FD_2,state
0,01001020100,Alabama,Autauga County,2025,2020,2020,1.0,1775.0,646.0,0.0,...,76979.0,1316.0,35.0,100.0,1.0,1.0,0.0,1080.0,1.0,01
1,01001020200,Alabama,Autauga County,2025,2020,2020,1.0,2055.0,723.0,0.0,...,62331.0,1148.0,28.0,143.0,1.0,0.0,0.0,167.0,0.0,01
2,01001020300,Alabama,Autauga County,2025,2020,2020,1.0,3216.0,1291.0,0.0,...,84191.0,2350.0,84.0,111.0,1.0,1.0,0.0,612.0,1.0,01
3,01001020400,Alabama,Autauga County,2025,2020,2020,1.0,4246.0,1707.0,0.0,...,81136.0,2580.0,114.0,78.0,1.0,0.0,0.0,14.0,0.0,01
4,01001020501,Alabama,Autauga County,2025,2020,2020,1.0,4322.0,1778.0,0.0,...,108233.0,4114.0,9.0,53.0,1.0,0.0,0.0,102.0,0.0,01


In [24]:
df_2025.isna().sum()

,0
tract_id,0
state_name,0
County,0
meta_dataset_version,0
meta_population_census_year,0
meta_tract_vintage,0
clean__Urban,0
clean__population_total,0
clean__occupied_housing_units_total,0
clean__GroupQuartersFlag,271


In [25]:
df_2025["clean__GroupQuartersFlag"] = df_2025["clean__GroupQuartersFlag"].fillna(df_2025["clean__GroupQuartersFlag"].median())
df_2025["clean__PCTGQTRS"] = df_2025["clean__PCTGQTRS"].fillna(df_2025["clean__PCTGQTRS"].median())

df_2025["clean__PovertyRate"] = df_2025["clean__PovertyRate"].fillna(df_2025["clean__PovertyRate"].median())
df_2025["clean__MedianFamilyIncome"] = df_2025["clean__MedianFamilyIncome"].fillna(df_2025["clean__MedianFamilyIncome"].median())

df_2025["clean__TractLOWI"] = df_2025["clean__TractLOWI"].fillna(df_2025["clean__TractLOWI"].median())
df_2025["clean__TractHUNV"] = df_2025["clean__TractHUNV"].fillna(df_2025["clean__TractHUNV"].median())
df_2025["clean__TractSNAP"] = df_2025["clean__TractSNAP"].fillna(df_2025["clean__TractSNAP"].median())

df_2025["clean__LATracts_half"] = df_2025["clean__LATracts_half"].fillna(df_2025["clean__LATracts_half"].median())
df_2025["clean__LATracts1"] = df_2025["clean__LATracts1"].fillna(df_2025["clean__LATracts1"].median())
df_2025["clean__LATractsVehicle_20"] = df_2025["clean__LATractsVehicle_20"].fillna(df_2025["clean__LATractsVehicle_20"].median())

df_2025["FD_2"] = df_2025["FD_2"].fillna(0)

In [26]:
df_2025.isna().sum()

,0
tract_id,0
state_name,0
County,0
meta_dataset_version,0
meta_population_census_year,0
meta_tract_vintage,0
clean__Urban,0
clean__population_total,0
clean__occupied_housing_units_total,0
clean__GroupQuartersFlag,0


## Predicting on Test Data

In [27]:
raw_test = pipe.predict(df_2025[model_cols])

output_test_results_df = df_2025.copy()
output_test_results_df["cluster"] = pd.Series(raw_test, index=df_2025.index).map(bundle["risk_map"])
output_test_results_df["risk"] = output_test_results_df["cluster"].map(bundle["risk_names"])

# adding state in second col of output file, first 2 digits of the tract_id
col = output_test_results_df.pop("state")
output_test_results_df.insert(1, col.name, col)

output_test_results_df.head()


,tract_id,state,state_name,County,meta_dataset_version,meta_population_census_year,meta_tract_vintage,clean__Urban,clean__population_total,clean__occupied_housing_units_total,...,clean__TractLOWI,clean__TractHUNV,clean__TractSNAP,clean__LATracts_half,clean__LATracts1,clean__LATractsVehicle_20,clean__low_access_population_1_20,FD_2,cluster,risk
0,01001020100,01,Alabama,Autauga County,2025,2020,2020,1.0,1775.0,646.0,...,1316.0,35.0,100.0,1.0,1.0,0.0,1080.0,1.0,2,mid
1,01001020200,01,Alabama,Autauga County,2025,2020,2020,1.0,2055.0,723.0,...,1148.0,28.0,143.0,1.0,0.0,0.0,167.0,0.0,2,mid
2,01001020300,01,Alabama,Autauga County,2025,2020,2020,1.0,3216.0,1291.0,...,2350.0,84.0,111.0,1.0,1.0,0.0,612.0,1.0,2,mid
3,01001020400,01,Alabama,Autauga County,2025,2020,2020,1.0,4246.0,1707.0,...,2580.0,114.0,78.0,1.0,0.0,0.0,14.0,0.0,0,low
4,01001020501,01,Alabama,Autauga County,2025,2020,2020,1.0,4322.0,1778.0,...,4114.0,9.0,53.0,1.0,0.0,0.0,102.0,0.0,0,low


In [28]:
print(output_test_results_df['risk'].value_counts())

risk
mid     39512
low     24014
high    20593
Name: count, dtype: int64


In [29]:
output_test_results_df.head()

,tract_id,state,state_name,County,meta_dataset_version,meta_population_census_year,meta_tract_vintage,clean__Urban,clean__population_total,clean__occupied_housing_units_total,...,clean__TractLOWI,clean__TractHUNV,clean__TractSNAP,clean__LATracts_half,clean__LATracts1,clean__LATractsVehicle_20,clean__low_access_population_1_20,FD_2,cluster,risk
0,01001020100,01,Alabama,Autauga County,2025,2020,2020,1.0,1775.0,646.0,...,1316.0,35.0,100.0,1.0,1.0,0.0,1080.0,1.0,2,mid
1,01001020200,01,Alabama,Autauga County,2025,2020,2020,1.0,2055.0,723.0,...,1148.0,28.0,143.0,1.0,0.0,0.0,167.0,0.0,2,mid
2,01001020300,01,Alabama,Autauga County,2025,2020,2020,1.0,3216.0,1291.0,...,2350.0,84.0,111.0,1.0,1.0,0.0,612.0,1.0,2,mid
3,01001020400,01,Alabama,Autauga County,2025,2020,2020,1.0,4246.0,1707.0,...,2580.0,114.0,78.0,1.0,0.0,0.0,14.0,0.0,0,low
4,01001020501,01,Alabama,Autauga County,2025,2020,2020,1.0,4322.0,1778.0,...,4114.0,9.0,53.0,1.0,0.0,0.0,102.0,0.0,0,low


In [30]:
#save result

output_test_results_df.to_csv("/content/drive/MyDrive/Food_Desert_Data/model_output_2025.csv", index=False)